# 在线RAG

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

C:\Users\yanchen\AppData\Local\Temp\ipykernel_36548\3182895319.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


### 加载训练好的 BM25 与向量数据库

In [4]:
# 加载向量数据库
# embedding_model_path = '/data/LLM/maple77/xiaobu-embedding-v2'
embedding_model_path = r"C:\Users\yanchen\.cache\huggingface\hub\models--BAAI--bge-small-zh-v1.5\snapshots\7999e1d3359715c523056ef9478215996d62a620"

embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_path,  
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

vectorstore = Chroma(
    persist_directory="./rag_cache/chroma_db",
    # 必须传入相同的 Embedding 实例
    embedding_function=embeddings     
)


# 加载 BM25
import pickle
load_path = './rag_cache/bm25_model.pkl'
with open(load_path, 'rb') as f:
    bm25_model = pickle.load(f)

### 编写进行 BM25 搜索函数，返回最相关的 10 条（0 分除外）

In [5]:
import jieba

def bm25_search(question_text, bm25_model):
    doc_scores = bm25_model.get_scores(jieba.lcut(question_text))
    scores_dict = {pos: soc for pos, soc in enumerate(doc_scores)}
    
    # 过滤掉是0分的,然后排序
    scores_dict_filter0 = dict(filter(lambda x: x[1] != 0, scores_dict.items()))

    if scores_dict_filter0:
        sorted_scores = sorted(scores_dict_filter0.items(), key=lambda item: item[1], reverse=True)
        nm25_max_score_page_idx_10 = [idx for idx, score in sorted_scores[:10]]
    else:
        nm25_max_score_page_idx_10 = []
    return nm25_max_score_page_idx_10

C:\Users\yanchen\Desktop\agent\.venv\Lib\site-packages\jieba\__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
C:\Users\yanchen\Desktop\agent\.venv\Lib\site-packages\jieba\__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
C:\Users\yanchen\Desktop\agent\.venv\Lib\site-packages\jieba\finalseg\__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
C:\Users\yanchen\Desktop\agent\.venv\Lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### 用户提出问题，对问题进行 BM25 检索

In [7]:
question_text = '什么情况会被退学'

# 返回最相关的 10 条 id（0 分除外）
nm25_max_score_page_idx_10 = bm25_search(question_text, bm25_model)

loader = TextLoader(
    "./普通高等学校学生管理规定.txt",
    encoding="utf-8"
)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # 每个文本块的大小
    chunk_overlap=50,  # 文本块之间的重叠部分
)
doc_splits = text_splitter.split_documents(documents)

# 根据 id 找到对应片段
bm25_search_res = [doc_splits[index].page_content for index in nm25_max_score_page_idx_10]
bm25_search_res

['第五十五条 在对学生作出处分或者其他不利决定之前，学校应当告知学生作出决定的事实、理由及依据，并告知学生享有陈述和申辩的权利，听取学生的陈述和申辩。\n\u3000\u3000处理、处分决定以及处分告知书等，应当直接送达学生本人，学生拒绝签收的，可以以留置方式送达；已离校的，可以采取邮寄方式送达；难于联系的，可以利用学校网站、新闻媒体等以公告方式送达。\n\u3000\u3000第五十六条 对学生作出取消入学资格、取消学籍、退学、开除学籍或者其他涉及学生重大利益的处理或者处分决定的，应当提交校长办公会或者校长授权的专门会议研究决定，并应当事先进行合法性审查。\n\u3000\u3000第五十七条 除开除学籍处分以外，给予学生处分一般应当设置6到12个月期限，到期按学校规定程序予以解除。解除处分后，学生获得表彰、奖励及其他权益，不再受原处分的影响。\n\u3000\u3000第五十八条 对学生的奖励、处理、处分及解除处分材料，学校应当真实完整地归入学校文书档案和本人档案。\n\u3000\u3000被开除学籍的学生，由学校发给学习证明。学生按学校规定期限离校，档案由学校退回其家庭所在地，户口应当按照国家相关规定迁回原户籍地或者家庭户籍所在地。\n\u3000\u3000第六章 学生申诉',
 '第四章 校园秩序与课外活动\n\u3000\u3000第三十九条 学校、学生应当共同维护校园正常秩序，保障学校环境安全、稳定，保障学生的正常学习和生活。\n\u3000\u3000第四十条 学校应当建立和完善学生参与管理的组织形式，支持和保障学生依法、依章程参与学校管理。\n\u3000\u3000第四十一条 学生应当自觉遵守公民道德规范，自觉遵守学校管理制度，创造和维护文明、整洁、优美、安全的学习和生活环境，树立安全风险防范和自我保护意识，保障自身合法权益。\n\u3000\u3000第四十二条 学生不得有酗酒、打架斗殴、赌博、吸毒，传播、复制、贩卖非法书刊和音像制品等违法行为；不得参与非法传销和进行邪教、封建迷信活动；不得从事或者参与有损大学生形象、有悖社会公序良俗的活动。\n\u3000\u3000学校发现学生在校内有违法行为或者严重精神疾病可能对他人造成伤害的，可以依法采取或者协助有关部门采取必要措施。\n\u3000\u3000第四十三条 学校应当坚持教育与宗教相

### 进行向量检索

In [8]:
# 返回最相关的 5 个
docs = vectorstore.similarity_search_with_score(question_text, k=5)

for doc in docs:
    print(doc[-1])

vec_search_res = [doc[0].page_content for doc in docs]

0.48158690333366394
0.6721349954605103
0.6852966547012329
0.6964058876037598
0.7289552688598633


In [9]:
vec_search_res #向量数据库会存原始文本、对应向量

['第五节 退学\n\u3000\u3000第三十条 学生有下列情形之一，学校可予退学处理：\n\u3000\u3000（一）学业成绩未达到学校要求或者在学校规定的学习年限内未完成学业的；\n\u3000\u3000（二）休学、保留学籍期满，在学校规定期限内未提出复学申请或者申请复学经复查不合格的；\n\u3000\u3000（三）根据学校指定医院诊断，患有疾病或者意外伤残不能继续在校学习的；\n\u3000\u3000（四）未经批准连续两周未参加学校规定的教学活动的；\n\u3000\u3000（五）超过学校规定期限未注册而又未履行暂缓注册手续的；\n\u3000\u3000（六）学校规定的不能完成学业、应予退学的其他情形。\n\u3000\u3000学生本人申请退学的，经学校审核同意后，办理退学手续。\n\u3000\u3000第三十一条 退学学生，应当按学校规定期限办理退学手续离校。退学的研究生，按已有毕业学历和就业政策可以就业的，由学校报所在地省级毕业生就业部门办理相关手续；在学校规定期限内没有聘用单位的，应当办理退学手续离校。\n\u3000\u3000退学学生的档案由学校退回其家庭所在地，户口应当按照国家相关规定迁回原户籍地或者家庭户籍所在地。\n\u3000\u3000第六节 毕业与结业',
 '第四节 休学与复学\n\u3000\u3000第二十五条 学生可以分阶段完成学业，除另有规定外，应当在学校规定的最长学习年限（含休学和保留学籍）内完成学业。\n\u3000\u3000学生申请休学或者学校认为应当休学的，经学校批准，可以休学。休学次数和期限由学校规定。\n\u3000\u3000第二十六条 学校可以根据情况建立并实行灵活的学习制度。对休学创业的学生，可以单独规定最长学习年限，并简化休学批准程序。\n\u3000\u3000第二十七条 新生和在校学生应征参加中国人民解放军（含中国人民武装警察部队），学校应当保留其入学资格或者学籍至退役后2年。\n\u3000\u3000学生参加学校组织的跨校联合培养项目，在联合培养学校学习期间，学校同时为其保留学籍。\n\u3000\u3000学生保留学籍期间，与其实际所在的部队、学校等组织建立管理关系。\n\u3000\u3000第二十八条 休学学生应当办理手续离校。学生休学期间，学校应为其保留学籍，但不享受在

### 创建 RAG Agent

In [11]:
from langchain.agents import create_agent
from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(model="deepseek-chat",
                api_key = '',
                base_url = 'https://api.deepseek.com')

system_prompt = """
你是一个专业的问答助手。请根据以下参考文档回答用户的问题。
如果参考文档中没有相关信息，请诚实地说不知道，不要编造答案。
"""

rag_agent = create_agent(model=model,
                     system_prompt = system_prompt)

### 根据知识库进行回答的结果

In [12]:
merge_query = '''
参考文档：{context}

用户问题：{question}
'''

context = "\n\n".join(list(set(bm25_search_res + vec_search_res)))
rag_query = merge_query.format(context=context, question=question_text)

response = rag_agent.invoke(
    {"messages": [{"role": "user", "content": rag_query}]}
)

print(response['messages'][-1].content)

根据参考文档第三十条，学生有下列情形之一，学校可予退学处理：

1. 学业成绩未达到学校要求或者在学校规定的学习年限内未完成学业的；
2. 休学、保留学籍期满，在学校规定期限内未提出复学申请或者申请复学经复查不合格的；
3. 根据学校指定医院诊断，患有疾病或者意外伤残不能继续在校学习的；
4. 未经批准连续两周未参加学校规定的教学活动的；
5. 超过学校规定期限未注册而又未履行暂缓注册手续的；
6. 学校规定的不能完成学业、应予退学的其他情形。

此外，学生本人申请退学的，经学校审核同意后，也会办理退学手续。


# 笔记
传统的RAG+Agent，其实主要是RAG中检索功能在发力，Agent没有起到什么作用